In [4]:
import os, sys
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
import shutil

In [5]:
# determine successful mutants - to show whether run was successful or not


# target_mutants = dict()
# path_to_datasets = "../Data/Protein_Gym_Datasets/"
# goal_th = 0.95
# for dataset in datasets:
#     df = pd.read_csv(os.path.join(path_to_datasets, f"{dataset}.csv"))
#     mutants = df["mutant"].to_numpy()
#     scores = df["Norm_Score_1"].to_numpy()
#     target_mutants[dataset] = set(mutants[scores >= goal_th])
    
    
# print()
# for key in target_mutants.keys():
#     print(f"{key}: {target_mutants[key]}")

datasets = ["YAP1", "GFP", "RASK", "PHOT"]
embedding_types = ["otf", "plm", "mpnn"]
target_mutants = {
"YAP1": ['D170Q:S183R', 'D170K:D194N'],
"GFP": ['K3R:G4S:L178H:Q184R', 'K3E:T230S', 'E5V:N121S:K140R', 'E5D:T9A:I171T', 'E6K:D76N:N105Y', 'F8L:D19N:N105Y:I171V', 'F8L:Q80R', 'T9A:I171V', 'V11A:D173N:S202I', 'I14V:T43A', 'L15M:M153V', 'H25R:I167V:G191S:N198D', 'H25R:E124K:K156R', 'H25Q:T59S:I171T', 'S28R:T97A:K107Q:I188V', 'E32G:E132A', 'E34A:T38S:T62A:V68M:I171T', 'T38A:K52R:N159D:K162E', 'T38N:S202R', 'T38S:K41R:N105S', 'Y39H:N146D', 'Y39H:N135S', 'Y39H:F99S:K126R:I171V', 'Y39N:N105I', 'Y39N:K158R', 'Y39N:T62A:Q157R', 'K41R:K158E:M233V', 'T43S:K107R:Q157H', 'T43S:K156E:K166R:I171V', 'T43S:D180N', 'F46Y:I47V', 'I47V:I161V', 'T62A:E111V:I161L', 'T62A:K131E:N170D', 'V68M:K131R:N159D:Q184R', 'F71L:E90G:K166E:N212D', 'S72G:N105S:Y182C:T230P', 'S72G:T203I:L236Q', 'R73L', 'D76V:H81L', 'D76G:I128T', 'H77Y:T97A', 'A87T:K158R:I171L', 'E90G:K158R:N198D', 'E90D:F99C:N146D:Q177L', 'E90A:V163A:N164S:N198D:N212S', 'T97A:N105Y:T225A', 'K101R:D180E:S205T', 'K101R:K214E', 'N105Y:G174S', 'Y106F:K113R:N198S', 'K107R:V163G:N212S', 'E111V:V176A:A227T', 'N121S:M153T:I171V', 'I128V:N146D', 'I128T:K158E', 'D129G:M233V', 'K131R:N170S', 'N135D:T203I', 'E142V:Q177L', 'N146S:K156E:K214E', 'S147N:T230P', 'K156E:Q184L', 'Q157R:G191S:V219L', 'K158G', 'K158R:N198D:K214E', 'K158R:Q184L', 'K158R:K162R:I171T:K214R', 'I161V:K166R', 'V163A:N164D:E172G:K214E', 'V163A:F223Y:I229V', 'V163A:V176A:Y182C:Q184R', 'V163A:S175R', 'V163E:Q184L', 'I171V:Q204L:L236V', 'S175T', 'Q184R:T186A', 'P192H:V193A:K214N'],
"RASK": ['A66P:H95N', 'A66P:R68M', 'R149V:I187P'],
"PHOT": ['R59D']
}

In [6]:
for embedding_type in embedding_types:
    results_dir = f"MLDE_Benchmark_{embedding_type}"
    # finished ensemble runs
    results_folders = os.listdir(results_dir)
    finished_folders = [f for f in results_folders if f.split("/")[-1].startswith("fin")]

    if embedding_type == "otf":
        total_runs = 19200 
    elif embedding_type == "plm":
        total_runs = 9600
    elif embedding_type == "mpnn":
        total_runs = 4800

    print(f"Finished ensemble {embedding_type} runs: {len(finished_folders)} / {total_runs} | {round(len(finished_folders) / total_runs * 100, 2)}%")

Finished ensemble otf runs: 18175 / 19200 | 94.66%
Finished ensemble plm runs: 8154 / 9600 | 84.94%
Finished ensemble mpnn runs: 3436 / 4800 | 71.58%


In [7]:
# build dataframe for results | rewritten
runs_data = []
faulty_finished_runs = []
for i, embedding_type in enumerate(embedding_types):
    results_folder_str= f"MLDE_Benchmark_{embedding_type}"
    results_folders = os.listdir(results_folder_str)
    with tqdm(total=len(results_folders), desc=f"Evaluating runs for {embedding_type} | {i+1}/{len(embedding_types)}") as pbar:
        for run in results_folders:
            run_id = str(run)

            try:
                #determine status: finished vs running
                status = "running"        
                if run_id.startswith("fin_"):
                    status = "finished"
                    run_id = run_id.replace("fin_", "")
                    
                dataset = run_id.split("_")[0]
                model = "_".join(run_id.split("_")[-10:-8]) if run_id.split("_")[-10] in ["elastic"] else run_id.split("_")[-9]
                n_Gain = int(run_id.split("_")[-8].strip("n_Gain"))
                n_Start = int(run_id.split("_")[-7].strip("n_Start"))
                run_number = int(run_id.split("_")[-1].strip("run"))
                embedding = "_".join(run_id.split("_")[1:3]) if run_id.split("_")[1] in ["esmc", "prot", "prost","one", "mpnn"] else run_id.split("_")[1]
                result = "n/a"
                
                
            except Exception as e:
                print(f"Error processing run_id: {run_id} | error: {e}")
                continue        
                #read performance.csv to determine the last documented cycle
                
                
            try:
                performance_file = os.path.join(results_folder_str, run, "performance.csv")
                with open(performance_file) as pf:
                    performances_entries = pf.readlines()
                    
                if len(performances_entries) > 1:
                    last_cycle = int(performances_entries[-1].split(",")[0])
                
                else: # no performances documented, allthough benchmark finished -> model not suitable, benchmark failed
                    last_cycle = 0
                    
                performances_found = True
            except FileNotFoundError:
                performances_found = False
                pass
                # run must be within the first cycle without any results yet
            
            #determine result for finished files - success vs failure
            if status == "finished":
                
                if performances_found: 
                    pass # last_cycle is last documented cycle from performance.csv
                    
                if not performances_found:      
                    # better check available last checkoints
                    last_cycle = len(os.listdir(os.path.join(results_folder_str, run,"checkpoints")))
                
                final_checkpoint_file = os.path.join(results_folder_str, run, f"checkpoints/cycle_{last_cycle}_checkpoint.txt")
                
                try:
                    with open(final_checkpoint_file, "r") as f:
                        checkpoint_lines = f.readlines()
                    try:
                        identified_mutants = checkpoint_lines[-1].replace(f"cycle {last_cycle} scored_mutants: ", "").strip().split(",")[:] # last line contains all identified mutants
                    except IndexError:
                        print(f"suspicious checkpoint file: {final_checkpoint_file} | content: {checkpoint_lines}")
                    for mutant in identified_mutants:
                        if mutant in target_mutants[dataset]:
                            result = "success"
                            break
                        else:
                            result = "failure"
                except FileNotFoundError:
                    result = "failure"
            if last_cycle == 0 and status == "finished": # no performance documented, no checkpoints available? remove folder and start again
                if result == "failure":
                    faulty_finished_runs.append([run_id])

            if status == "running":  # current cycle is one more than available checkpoints  
                result = "open"
            
            runs_data.append([run_id, status, result, dataset, model, embedding, n_Gain, n_Start, run_number, last_cycle])
            pbar.update(1)
runs_df = pd.DataFrame(runs_data, columns=["run_id", "status", "result", "dataset", "model", "embedding", "n_Gain", "n_Start", "run", "last_cycle"])
faulty_finished_runs_df = pd.DataFrame(faulty_finished_runs, columns=["run_id"])

print(f"Runs proceeded. {len(faulty_finished_runs_df)} have been marked for renaming/removal from finished runs due to missing files.")

Evaluating runs for otf | 1/3: 100%|██████████| 18895/18895 [02:35<00:00, 121.62it/s]
Evaluating runs for plm | 2/3: 100%|██████████| 9539/9539 [01:08<00:00, 139.72it/s]
Evaluating runs for mpnn | 3/3: 100%|██████████| 3820/3820 [00:30<00:00, 125.75it/s]

Runs proceeded. 0 have been marked for renaming/removal from finished runs due to missing files.


In [8]:
test = faulty_finished_runs_df["run_id"].to_list()[:10]
for r in test:
    print(r)

In [9]:
skipped_runs = []
for run_id in faulty_finished_runs_df["run_id"].to_list():

    found = False
    path_to_script = "/work2/ak45befu-MAP/MAP_URZ/Results"
    for embedding_type in embedding_types:
        if not found:
            results_folder_str= f"MLDE_Benchmark_{embedding_type}"

            old_name = os.path.join(results_folder_str, f"fin_{run_id}")
            new_name = os.path.join(results_folder_str, f"{run_id}")
            
            if os.path.exists(old_name):
                found = True
                shutil.move(old_name, new_name)
                #print(f"Renamed {old_name} to {new_name} to unmark it as finished.")

        
    if not found:
        skipped_runs.append(run_id)
        print("run not found for renaming: ", run_id)
        


            
print(f"Renaming completed.")
if len(skipped_runs) > 0:
    print(f"{len(skipped_runs)} runs could not be found for renaming and should be checked manually: {skipped_runs}")

Renaming completed.


In [10]:
# Status for bash scripts:
embeddings = [["one_hot", "blosum50", 'blosum90',"georgiev"],["esmc_600m", "esmc_300m"],["prost_t5"],["mpnn_probs"]] 
runs = [[1,2,3,4,5],[6,7,8,9,10]]

try:
    for emb in embeddings:
        for r in runs:

            bash_finished = runs_df[(runs_df["status"] == "finished") 
                                    & (runs_df['embedding'].isin(emb))
                                    & (runs_df['run'].isin(r))
                                ]
            if len(bash_finished) != 0:
                print(f"{emb} | runs {r} : ", len(bash_finished))

except Exception as e:
    pass

['one_hot', 'blosum50', 'blosum90', 'georgiev'] | runs [1, 2, 3, 4, 5] :  9081
['one_hot', 'blosum50', 'blosum90', 'georgiev'] | runs [6, 7, 8, 9, 10] :  9094
['esmc_600m', 'esmc_300m'] | runs [1, 2, 3, 4, 5] :  2210
['esmc_600m', 'esmc_300m'] | runs [6, 7, 8, 9, 10] :  2190
['prost_t5'] | runs [1, 2, 3, 4, 5] :  1874
['prost_t5'] | runs [6, 7, 8, 9, 10] :  1879
['mpnn_probs'] | runs [1, 2, 3, 4, 5] :  1718
['mpnn_probs'] | runs [6, 7, 8, 9, 10] :  1719


In [11]:
# Get unique run_ids that appear more than once
duplicate_run_ids = runs_df[runs_df.duplicated(subset=['run_id'], keep=False)][['run_id', 'embedding']]
print(f"Total unique run_ids with duplicates: {len(duplicate_run_ids)}\n")
print("Run IDs that appear more than once:")
print("="*80)
print(duplicate_run_ids.sort_values('run_id'))

Total unique run_ids with duplicates: 2

Run IDs that appear more than once:
                                                  run_id   embedding
21878  GFP_mpnn_probs_linear_nGain100_nStart500_ddG_t...  mpnn_probs
31011  GFP_mpnn_probs_linear_nGain100_nStart500_ddG_t...  mpnn_probs


In [12]:
# remove dublicate runs where finished version exists
import shutil

with tqdm(total=len(duplicate_run_ids), desc="Removing unfinished duplicates") as pbar:
    for duplicate in duplicate_run_ids.index:
        if duplicate["embedding"] in ["one_hot", "blosum50", 'blosum90',"georgiev"]:
            emb_type = "otf"
        elif duplicate["embedding"] in ["esmc_600m", "esmc_300m","prost_t5"]:
            emb_type = "plm"
        elif duplicate["embedding"] in ["mpnn_probs"]:
            emb_type = "mpnn"

        results_dir = f"MLDE_Benchmark_{emb_type}"
        path_finished_run = os.path.join(results_dir, f"fin_{duplicate}")
        path_unfinished_run = os.path.join(results_dir, duplicate)
        if os.path.exists(path_finished_run):
            if os.path.exists(path_unfinished_run):
                try:
                    shutil.rmtree(path_unfinished_run)
                    pbar.update(1)
                except Exception as e:
                    pass

Removing unfinished duplicates:   0%|          | 0/2 [00:00<?, ?it/s]


TypeError: 'int' object is not subscriptable

In [ ]:
# identify false negative rf runs
# all runs with rf as algortihm, failed in after the first cycle
# if not failed in the first cycle, the algorithm must somehow worked before. There it should work as well with more data in the same setup
# -> probably failed due to parameter key error in the code -> undo renaming of the finished folder to not finished, \
# so that the run can be continued and decide again, whether the setup is suitable or not 

failed_rf_runs = runs_df[(runs_df['status'] == "finished")
                            # & (runs_df['model'] == "rf")
                            # & (runs_df['embedding'] == "mpnn_probs")
                            & (runs_df['result'] == 'failure')
                            # & (runs_df['last_cycle'] == 1)
]

print(f"Total failed rf runs: {len(failed_rf_runs)}")
print("    of which failed in the first cycle: ", len(failed_rf_runs[failed_rf_runs['last_cycle'] == 1]))
print("    of which failed after the first cycle: ", len(failed_rf_runs[failed_rf_runs['last_cycle'] == 2]))
# if len(failed_rf_runs) > 0:
#     print(f"Failed rf runs: {len(failed_rf_runs)}")
#     for run_id in failed_rf_runs["run_id"].tolist():
#         print(f"  - {run_id}")

# printing those who failed in the first cycle:
failed_in_first_cycle = failed_rf_runs[failed_rf_runs['last_cycle'] == 1]
print("Failed in the first cycle:")
for run_id in failed_in_first_cycle["run_id"].tolist():
    print(f"  - {run_id}")


rename = False
# adapt script to check in all embedding type - folders
if rename:
    with tqdm(total=len(failed_rf_runs["run_id"].tolist()), desc=f"renaming faulty rf_runs") as pbar:
        for failed_run in failed_rf_runs["run_id"].tolist():
            old_name = os.path.join(results_dir, f"fin_{failed_run}")
            new_name = os.path.join(results_dir, f"{failed_run}")
            if os.path.exists(old_name):
                try:
                    shutil.move(old_name, new_name)
                    pbar.update(1)
                except Exception as e:
                    print(f"Error renaming {old_name} to {new_name}: {e}")

Total failed rf runs: 14338
    of which failed in the first cycle:  42
    of which failed after the first cycle:  23
Failed rf runs: 14338
  - GFP_georgiev_fnn_nGain20_nStart200_ddG_top_start0.15_nCycles40_normedScores_run7
  - PHOT_one_hot_linear_nGain100_nStart3000_ddG_top_start0.15_nCycles40_normedScores_run2
  - YAP1_one_hot_rf_nGain20_nStart1000_ddG_top_start0.15_nCycles40_normedScores_run10
  - PHOT_blosum50_linear_nGain20_nStart500_ddG_top_start0.15_nCycles40_normedScores_run10
  - GFP_one_hot_elastic_net_nGain10_nStart100_ddG_top_start0.15_nCycles40_normedScores_run1
  - RASK_blosum90_fnn_nGain20_nStart100_ddG_top_start0.15_nCycles40_normedScores_run10
  - GFP_blosum50_rf_nGain10_nStart200_ddG_top_start0.15_nCycles40_normedScores_run10
  - PHOT_georgiev_lightgbm_nGain50_nStart1000_ddG_top_start0.15_nCycles40_normedScores_run7
  - PHOT_one_hot_rf_nGain20_nStart1000_ddG_top_start0.15_nCycles40_normedScores_run1
  - YAP1_blosum50_rf_nGain20_nStart1000_ddG_top_start0.15_nCycles40

In [15]:
date = datetime.now().strftime("%d%m%Y")
file_name = f"runs_results_{str(date)}.csv"

In [ ]:
# Save runs_df to CSV file
runs_df.to_csv(file_name, index=False)
print(f"Saved runs_df to {file_name} ({len(runs_df)} rows)")

Saved runs_df to runs_results_17042026.csv (31616 rows)
